In [1]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
import numpy as np


In [2]:
df = pd.read_csv("data/restructured.csv")

In [3]:
df.head()

,roundup_title,topics,story_title,story_text,bias_label
0,"""Dream"" 50 Years Later",Civil Rights,Thousands Gather In D.C. To Mark 1963 Civil Ri...,People are assembling on the National Mall to ...,center
1,"""Dream"" 50 Years Later",Civil Rights,March In Washington To Continue Focus On Civil...,Alice Long planned months ago to use vacation ...,left
2,"""Dream"" 50 Years Later",Civil Rights,Remembering My Uncle's 'Dream',"Fifty years ago, a valiant group of people fro...",right
3,"""Good Shutdown"" in September?",Politics,"President Trump Calls for a ""Good Shutdown"" in...",President Donald Trump made a bold statement o...,right
4,"""Good Shutdown"" in September?",Politics,Trump: US ‘needs a good shutdown’,"President Trump on Tuesday called for a ""good ...",center


### Pre-Process

In [4]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

# Helper: map NLTK POS tags to WordNet POS
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default to noun

# Improved preprocessing
def preprocess(text):
    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Tokenize
    tokens = word_tokenize(text)

    # POS tagging and lemmatization
    tagged = pos_tag(tokens)
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

    return ' '.join(lemmatized)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alsei\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alsei\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\alsei\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alsei\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


#### Keep Special Characters for LSTM Text Classification?

Yes — for political bias detection using LSTM or BiLSTM, it's beneficial to **retain punctuation and special characters** like quotes (`'`, `"`) or ellipses (`...`).

##### Why?
- **Framing and tone** often depend on punctuation (e.g., irony, sarcasm).
- **Quotes** can imply indirectness or subjectivity — common in politically charged texts.
- LSTM models can learn **contextual meaning from raw character sequences**.

##### Tip:
Clean only unnecessary noise (e.g., HTML, broken encoding), but **don’t over-sanitize** your text — keep the "flavor" of how things are said.


In [5]:
# combine title and story into one column
df['combined'] = '[TITLE] ' + df['story_title'] + ' [TEXT] ' + df['story_text']


In [6]:
# apply preprocess function on story_title and story_text
df['combined'] = df['combined'].apply(preprocess)

### Check Balance

In [7]:
# check balance of the dataset
df['bias_label'].value_counts()

bias_label
left      8430
right     8375
center    7700
Name: count, dtype: int64

In [8]:
# remove rows where there is no center (group for roundup_title and check if there is a center)

# Count how many bias labels exist per roundup
grouped = df.groupby('roundup_title')['bias_label'].nunique()

# Filter only roundups that contain all 3 labels
valid_titles = grouped[grouped == 3].index

# Keep only rows with those titles
df = df[df['roundup_title'].isin(valid_titles)].copy()


In [9]:
# check balance of the dataset
df['bias_label'].value_counts()

bias_label
center    7564
left      7564
right     7563
Name: count, dtype: int64

### make all the same length by using padding

In [10]:
# splitinto tokens to count the number of tokens
df['token_len'] = df['combined'].apply(lambda x: len(x.split()))
# check the distribution of token lengths
max_len = int(np.percentile(df['token_len'], 95))
print("95th percentile max_len:", max_len)


95th percentile max_len: 133


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['combined'])

sequences = tokenizer.texts_to_sequences(df['combined'])

# add padding to the sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='pre', truncating='post')

In [12]:
print("Original sequence:", sequences[0])
print("Padded sequence:", padded_sequences[0])

Original sequence: [10, 578, 1276, 6, 248, 718, 3, 337, 8498, 803, 144, 457, 11, 72, 5, 4861, 8, 1, 108, 3829, 3, 4411, 1, 8077, 1633, 4, 2329, 3632, 1434, 817, 26, 8498, 457, 8, 186, 1019, 216, 2, 1, 4339, 14, 1, 4965, 65, 15, 2, 2947, 37, 319, 13, 306, 8499, 1, 803, 144, 1324, 4271, 110, 1, 6022, 945, 13785, 7, 1434, 26, 704, 2329, 3632, 1434, 3266, 18, 54, 226, 698, 117, 72, 3, 1503, 278, 26, 506, 207, 82, 1, 85, 2346, 1764, 1633, 319, 35, 5, 228, 23, 2, 732, 1435, 1189, 22, 1, 3884, 1641, 3, 1, 239, 40, 305, 2329, 3632, 1434, 817, 1641]
Padded sequence: [    0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0    10   578
  1276     6   248   718     3   337  8498   803   144   457    11    72
     5  4861     8     1   108  3829     3  4411     1  8077  1633     4
  2329  3632  1434   817    26  8498   457     8   186  1019   216     2
     1  4339    14     1  4965    65    15     2  2947    37   319    1

In [13]:
word_index = tokenizer.word_index
print({k: word_index[k] for k in list(word_index)[:20]})  # Show first 10

{'the': 1, 'a': 2, 'to': 3, 'of': 4, 'be': 5, 'in': 6, 'and': 7, 'on': 8, 's': 9, 'title': 10, 'text': 11, '’': 12, 'that': 13, 'for': 14, 'have': 15, 'trump': 16, 'it': 17, 'say': 18, 'president': 19, 'with': 20}


### use these sequences to create vectors

#### Trainable Embedding (baseline)

In [14]:
# convert word indices to word vectors
from tensorflow.keras.layers import Embedding

vocab_size = len(tokenizer.word_index) + 1  # +1 for padding token
embedding_dim = 100  # common choice, can try 50 or 300 too

embedding_baseline = Embedding(input_dim=vocab_size,    # size of vocabulary
                               output_dim=embedding_dim,# embedding dimension
                               input_length=max_len)        # max_len from earlier (133)

c:\Users\alsei\Documents\GitHub\NLP_Final\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


What This Code Does:

- Initializes a trainable embedding layer
- Converts word indices (from `padded_sequences`) into 100D vectors
- Learns embeddings **from scratch** during model training
- Uses fixed input length (`input_length = 133`) to match padded sequences

#### word2vec

In [15]:
import gensim.downloader as api

word2vec = api.load("word2vec-google-news-300")  # pretrained vectors

In [16]:
embedding_dim = 300  # Word2Vec GoogleNews uses 300D vectors
vocab_size = len(tokenizer.word_index) + 1
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in word2vec:
        embedding_matrix[i] = word2vec[word]
    # else: remains zero

In [17]:
from tensorflow.keras.layers import Embedding

# baseline Word2Vec (frozen)
embedding_word2vec_static = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=max_len,
    trainable=False
)

# fine-tuned Word2Vec
embedding_word2vec_trainable = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=max_len,
    trainable=True  # allow gradient updates
)


#### Summary: Using Pretrained Word2Vec Embeddings (GoogleNews)

This code sets up a non-trainable embedding layer using the **pretrained Word2Vec GoogleNews model** (300-dimensional vectors).

### prepare data for model training

In [18]:
# encode the target variable
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['bias_label'])

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, y, test_size=0.2, random_state=42, stratify=y
)


In [20]:
for i in range(3):  # show first 3 examples
    print(f"Sequence {i}:")
    print(padded_sequences[i])
    print(f"Label: {y[i]}")
    print("---")

Sequence 0:
[    0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0    10   578
  1276     6   248   718     3   337  8498   803   144   457    11    72
     5  4861     8     1   108  3829     3  4411     1  8077  1633     4
  2329  3632  1434   817    26  8498   457     8   186  1019   216     2
     1  4339    14     1  4965    65    15     2  2947    37   319    13
   306  8499     1   803   144  1324  4271   110     1  6022   945 13785
     7  1434    26   704  2329  3632  1434  3266    18    54   226   698
   117    72     3  1503   278    26   506   207    82     1    85  2346
  1764  1633   319    35     5   228    23     2   732  1435  1189    22
     1  3884  1641     3     1   239    40   305  2329  3632  1434   817
  1641]
Label: 0
---
Sequence 1:
[    0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0

In [21]:
# function to define the models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.regularizers import l2

def build_model(embedding_layer, lstm_units=32, dropout=0.2, dense_units=None, l2_reg=None):
    model = Sequential()
    model.add(embedding_layer)
    model.add(LSTM(lstm_units, dropout=dropout, recurrent_dropout=dropout))

    if dense_units and dense_units > 0:
        if l2_reg:
            model.add(Dense(dense_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        else:
            model.add(Dense(dense_units, activation='relu'))

    model.add(Dense(3, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [22]:
v1_baseline_LSTM = build_model(embedding_baseline)
v1_word2vec_LSTM = build_model(embedding_word2vec_static)

v2_baseline_LSTM = build_model(embedding_baseline, lstm_units=64, dropout=0.3, dense_units=128)
v2_word2vec_LSTM = build_model(embedding_word2vec_static, lstm_units=64, dropout=0.3, dense_units=128)

v3_baseline_LSTM = build_model(embedding_baseline, lstm_units=32, dropout=0.2, dense_units=0)
v3_word2vec_LSTM = build_model(embedding_word2vec_static, lstm_units=32, dropout=0.2, dense_units=0)

models on static word2vec

In [23]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print('v1_baseline_LSTM')
v1_baseline_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])
print('v1_word2vec_LSTM')
v1_word2vec_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])

print('v2_baseline_LSTM')
v2_baseline_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])
print('v2_word2vec_LSTM')
v2_word2vec_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])

print('v3_baseline_LSTM')
v3_baseline_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])
print('v3_word2vec_LSTM')
v3_word2vec_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])


v1_baseline_LSTM
Epoch 1/10
511/511 ━━━━━━━━━━━━━━━━━━━━ 42s 75ms/step - accuracy: 0.3501 - loss: 1.0973 - val_accuracy: 0.3893 - val_loss: 1.0912
Epoch 2/10
105/511 ━━━━━━━━━━━━━━━━━━━━ 28s 71ms/step - accuracy: 0.5247 - loss: 1.0365

KeyboardInterrupt: 

fine-tuned word2vec

In [ ]:
# include L2 regularization
v4_word2vec_LSTM = build_model(embedding_layer=embedding_word2vec_trainable,lstm_units=32,dropout=0.3,dense_units=64, l2_reg=0.01)

#include lighter l2 regularization
v5_word2vec_LSTM = build_model(embedding_layer=embedding_word2vec_trainable,lstm_units=32,dropout=0.3,dense_units=64, l2_reg=0.001)

In [ ]:
print('v4_word2vec_LSTM')
v4_word2vec_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])

print('v5_word2vec_LSTM')
v5_word2vec_LSTM.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, callbacks=[early_stop])

v4_word2vec_LSTM
Epoch 1/10
552/552 ━━━━━━━━━━━━━━━━━━━━ 81s 140ms/step - accuracy: 0.3409 - loss: 1.2639 - val_accuracy: 0.3738 - val_loss: 1.0947
Epoch 2/10
552/552 ━━━━━━━━━━━━━━━━━━━━ 77s 140ms/step - accuracy: 0.4185 - loss: 1.0793 - val_accuracy: 0.3728 - val_loss: 1.0969
Epoch 3/10
552/552 ━━━━━━━━━━━━━━━━━━━━ 78s 141ms/step - accuracy: 0.5655 - loss: 0.9542 - val_accuracy: 0.3544 - val_loss: 1.1955
v5_word2vec_LSTM
Epoch 1/10
552/552 ━━━━━━━━━━━━━━━━━━━━ 85s 145ms/step - accuracy: 0.3817 - loss: 1.1154 - val_accuracy: 0.3605 - val_loss: 1.1145
Epoch 2/10
552/552 ━━━━━━━━━━━━━━━━━━━━ 82s 149ms/step - accuracy: 0.5213 - loss: 1.0006 - val_accuracy: 0.3595 - val_loss: 1.1676
Epoch 3/10
552/552 ━━━━━━━━━━━━━━━━━━━━ 79s 144ms/step - accuracy: 0.6528 - loss: 0.8143 - val_accuracy: 0.3447 - val_loss: 1.2981
